# 07 — Circuit and hardware resources

Translates the QUBO interaction graph into logical cost-layer and connectivity metrics. No noise model is used.

In [ ]:
from __future__ import annotations

import pandas as pd

from oec_qaoa.benchmarks import (
    FROZEN_QAOA_BENCHMARKS,
    build_frozen_benchmark,
)
from oec_qaoa.circuit_resources import (
    connectivity_metrics,
    cost_layer_metrics,
)

TOPOLOGIES = ["all_to_all", "grid", "ring", "line"]

## Logical cost-layer structure

Each non-zero quadratic coefficient produces a `ZZ` interaction. The maximum interaction-graph degree gives a lower bound on parallel two-qubit slots; greedy edge coloring provides a constructive schedule.

In [ ]:
logical_rows = []

for benchmark in FROZEN_QAOA_BENCHMARKS:
    instance, _, model = build_frozen_benchmark(benchmark)
    metrics = cost_layer_metrics(model)

    logical_rows.append(
        {
            "benchmark_id": benchmark.benchmark_id,
            "instance": instance.name,
            **metrics.to_dict(),
        }
    )

logical = pd.DataFrame(logical_rows)
logical

At QAOA depth `p`, the logical number of pairwise cost interactions is `p N_ZZ`; the all-to-all cost-layer depth scales between `p Δ` and the greedy schedule depth.

## Representative connectivity graphs

The logical interaction graph is mapped to all-to-all, grid, ring and line topologies. We report direct-support fraction, mean and maximum shortest-path distance, and the routing-pressure proxy

\[
R_{\mathrm{dist}}=\sum_{(i,j)\in E_{\mathrm{QUBO}}}\max(d_{ij}-1,0).
\]

In [ ]:
rows = []

for benchmark in FROZEN_QAOA_BENCHMARKS:
    instance, _, model = build_frozen_benchmark(benchmark)

    for topology in TOPOLOGIES:
        metrics = connectivity_metrics(model, topology)
        rows.append(
            {
                "benchmark_id": benchmark.benchmark_id,
                "instance": instance.name,
                **metrics.to_dict(),
            }
        )

connectivity = pd.DataFrame(rows)
connectivity

## Benchmark comparison

Logical interaction complexity and variational performance are reported separately.

In [ ]:
summary = connectivity.merge(
    logical[
        [
            "benchmark_id",
            "logical_qubits",
            "zz_terms",
            "interaction_max_degree",
            "greedy_parallel_depth",
        ]
    ],
    on="benchmark_id",
    how="left",
)

summary[
    [
        "benchmark_id",
        "topology",
        "logical_qubits",
        "zz_terms",
        "interaction_max_degree",
        "greedy_parallel_depth",
        "directly_supported_fraction",
        "mean_interaction_distance",
        "max_interaction_distance",
        "distance_excess_sum",
    ]
]

## Scope

The connectivity quantities are topology-level resource indicators rather than compiled SWAP counts.